# `single_feature_model_performance`

## Purpose

Get performance of logistic regression and SVC models with a single input feature for each feature available in our dataset and store performance metrics. These metrics will be used in the feature selection process.

## Previous notebook

`define_train_test_split`

## Next notebook

`collinearity_analysis`

# Imports

In [1]:
import numpy as np
import pandas as pd

import os
og_dir = os.getcwd()
os.chdir('../jupyter')
import cutpoint_analysis
os.chdir(og_dir)

import prepare_data

import pickle
import time

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score, average_precision_score, RocCurveDisplay, accuracy_score

In [2]:
from set_env_vars import set_all_env_vars

set_all_env_vars()

# Load and process data

In [1]:
train_cohort_ll_imp = prepare_data.load_and_process_cohort('train', 'latest')
val_cohort_ll_imp = prepare_data.load_and_process_cohort('val', 'latest')

train_cohort_med_imp = prepare_data.load_and_process_cohort('train', 'median')
val_cohort_med_imp = prepare_data.load_and_process_cohort('val', 'median')

In [9]:
feature_cols = prepare_data.get_feature_cols(train_cohort_ll_imp)

In [10]:
'cardiac_arrest' in feature_cols

True

## Remove non-feature columns from dataframes

In [11]:
train_cohort_ll_imp = train_cohort_ll_imp[feature_cols + ['aki_72hrs_any']]
val_cohort_ll_imp = val_cohort_ll_imp[feature_cols + ['aki_72hrs_any']]

train_cohort_med_imp = train_cohort_med_imp[feature_cols + ['aki_72hrs_any']]
val_cohort_med_imp = val_cohort_med_imp[feature_cols + ['aki_72hrs_any']]

# Train models on one feature at a time

## Logistic regression

In [12]:
def get_metrics_for_single_feature_logreg_model(df, 
                                                input_feature_name, 
                                                target_feature_name='aki_72hrs_any',
                                                max_iter=10000,
                                                random_state=343):
    X = df[input_feature_name].to_numpy()
    y = df[target_feature_name].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

    model = LogisticRegression(class_weight='balanced', max_iter=max_iter, random_state=random_state)
    model.fit(X_train.reshape(-1, 1), y_train)

    y_score = model.predict_proba(X_test.reshape(-1, 1))
    auroc = roc_auc_score(y_test, y_score[:,1])
    auprc = average_precision_score(y_test, y_score[:,1])

    results_dict = {
        'input_feature': input_feature_name,
        'auroc': auroc,
        'auprc': auprc
    }

    return results_dict

In [13]:
for col in train_cohort_ll_imp.columns:
    na_count = train_cohort_ll_imp[col].isna().sum()
    if na_count > 0:
        print(col + ' NA count = %d' % na_count)

In [14]:
for col in train_cohort_med_imp.columns:
    na_count = train_cohort_med_imp[col].isna().sum()
    if na_count > 0:
        print(col + ' NA count = %d' % na_count)

### Latest lab & median imputation

In [16]:
for feature in feature_cols:
    filename = 'pickle/single_feature_logreg_metrics_ll_imp/' + feature + '.pickle'
    if not os.path.isfile(filename):
        print(feature)
        temp_results = get_metrics_for_single_feature_logreg_model(
            train_cohort_ll_imp, 
            feature, 
            target_feature_name='aki_72hrs_any',
            max_iter=10000,
            random_state=343
        )
        with open(filename, 'wb') as outfile:
            pickle.dump(temp_results, outfile)

cardiac_arrest


### Median imputation only

In [17]:
for feature in feature_cols:
    filename = 'pickle/single_feature_logreg_metrics_med_imp/' + feature + '.pickle'
    if not os.path.isfile(filename):
        print(feature)
        temp_results = get_metrics_for_single_feature_logreg_model(
            train_cohort_med_imp, 
            feature, 
            target_feature_name='aki_72hrs_any',
            max_iter=10000,
            random_state=343
        )
        with open(filename, 'wb') as outfile:
            pickle.dump(temp_results, outfile)

cardiac_arrest


### Save resulting performance metrics

In [18]:
logreg_results_list = []

for filename in os.listdir('pickle/single_feature_logreg_metrics_ll_imp'):
    try:
        with open('pickle/single_feature_logreg_metrics_ll_imp/'+filename, 'rb') as infile:
            logreg_results_list.append(pickle.load(infile))
    except:
        print('Exception for filename ' + filename)
        
logreg_results_df = pd.DataFrame(logreg_results_list)

logreg_results_df.to_csv('single_feature_logreg_performance_results_latest_lab_imp.csv', index=False)

In [19]:
logreg_med_imp_results_list = []

for filename in os.listdir('pickle/single_feature_logreg_metrics_med_imp'):
    try:
        with open('pickle/single_feature_logreg_metrics_med_imp/'+filename, 'rb') as infile:
            logreg_med_imp_results_list.append(pickle.load(infile))
    except:
        print('Exception for filename ' + filename)
        
logreg_med_imp_results_df = pd.DataFrame(logreg_med_imp_results_list)

logreg_med_imp_results_df.to_csv('single_feature_logreg_performance_results_median_imp.csv', index=False)

## SVC

In [20]:
def get_metrics_for_single_feature_svc_model(df, 
                                            input_feature_name, 
                                            target_feature_name='aki_72hrs_any',
                                            max_iter=1000,
                                            random_state=343):
    X = df[input_feature_name].to_numpy()
    y = df[target_feature_name].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

    model = SVC(class_weight='balanced', max_iter=max_iter, random_state=random_state, probability=True)
    model.fit(X_train.reshape(-1, 1), y_train)

    y_score = model.predict_proba(X_test.reshape(-1, 1))
    auroc = roc_auc_score(y_test, y_score[:,1])
    auprc = average_precision_score(y_test, y_score[:,1])

    results_dict = {
        'input_feature': input_feature_name,
        'model_type': 'SVC',
        'auroc': auroc,
        'auprc': auprc
    }

    return results_dict

### Latest lab & median imputation

In [21]:
svc_results_list = []

for feature in feature_cols:
    filename = 'pickle/single_feature_svc_metrics_ll_imp/' + feature + '.pickle'
    if not os.path.isfile(filename):
        print(feature)
        temp_results = get_metrics_for_single_feature_svc_model(
            train_cohort_ll_imp, 
            feature, 
            target_feature_name='aki_72hrs_any',
            max_iter=1000,
            random_state=343
        )
        with open(filename, 'wb') as outfile:
            pickle.dump(temp_results, outfile)
    svc_results_list.append(temp_results)
    
svc_results_df = pd.DataFrame(svc_results_list)

svc_results_df.to_csv('single_feature_svc_performance_results_latest_lab_imp.csv', index=False)

cardiac_arrest


/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


### Median imputation only

In [24]:
for feature in feature_cols:
    filename = 'pickle/single_feature_svc_metrics_med_imp/' + feature + '.pickle'
    if not os.path.isfile(filename):
        print(feature)
        temp_results = get_metrics_for_single_feature_svc_model(
            train_cohort_med_imp, 
            feature, 
            target_feature_name='aki_72hrs_any',
            max_iter=1000,
            random_state=343
        )
        with open(filename, 'wb') as outfile:
            pickle.dump(temp_results, outfile)

cardiac_arrest


/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


### Save resulting performance metrics

In [ ]:
svc_ll_imp_results_list = []

for filename in os.listdir('pickle/single_feature_svc_metrics_ll_imp'):
    try:
        with open('pickle/single_feature_svc_metrics_ll_imp/'+filename, 'rb') as infile:
            svc_ll_imp_results_list.append(pickle.load(infile))
    except:
        print('Exception for filename ' + filename)
        
svc_ll_imp_results_df = pd.DataFrame(svc_ll_imp_results_list)

svc_ll_imp_results_df.to_csv('single_feature_svc_performance_results_latest_lab_imp.csv', index=False)

In [25]:
svc_med_imp_results_list = []

for filename in os.listdir('pickle/single_feature_svc_metrics_med_imp'):
    try:
        with open('pickle/single_feature_svc_metrics_med_imp/'+filename, 'rb') as infile:
            svc_med_imp_results_list.append(pickle.load(infile))
    except:
        print('Exception for filename ' + filename)
        
svc_med_imp_results_df = pd.DataFrame(svc_med_imp_results_list)

svc_med_imp_results_df.to_csv('single_feature_svc_performance_results_median_imp.csv', index=False)